# Phase 2: MedGemma Local Calibration with Rolling Memory

## PCDC Framework Overview

This notebook implements VIPS-aligned Patient-Centered Dementia Care (PCDC) logic for on-device dementia care assistance using MedGemma language model.

**VIPS Framework:**
- **Value:** Listen to what matters to the person
- **Individualize:** Tailor approaches to their unique life story  
- **Perspective:** Understand the person's viewpoint (not assume behavior is "behavioral problem")
- **Social Psychology:** Use social engagement principles to reduce agitation

**Key Features:**
- Rolling memory via `session_memory.json` for cross-turn context
- Deterministic inference (no sampling) for reproducibility
- Auto-summarization every 5 turns to prevent context dilution
- Structured JSON output for iOS UI consumption
- VIPS adherence validation on synthetic turns

## Section 1: Import Libraries and Load Session Memory Structure

In [ ]:
import json
import csv
import os
import sys
from pathlib import Path
from typing import Dict, List, Any, Tuple
from datetime import datetime

# Machine learning imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
exports_dir = project_root / 'exports'
assets_dir = project_root / 'assets'

print(f"Project root: {project_root}")
print(f"Exports dir: {exports_dir}")
print(f"PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

# Session memory schema - defines the persistent state across turns
SESSION_MEMORY_SCHEMA = {
    "patient_profile": "",  # e.g., "Mary, 82, widow, loved gardening"
    "session_summary": "",  # e.g., "10min: calmer after garden talk; keywords: roses"
    "current_trend": "",    # e.g., "stabilizing_calm" or "escalating_agitation"
    "keywords": [],         # e.g., ["roses", "garden"]
    "last_nudge": "",       # e.g., "Gently affirm garden memories"
    "agritation_history": [],  # List of agitation scores from recent turns
    "turn_count": 0         # Total turn counter for auto-summarization trigger
}

def load_session_memory(json_path: Path) -> Dict[str, Any]:
    """
    Load session memory from JSON file.
    If file doesn't exist, initialize with schema.
    
    Args:
        json_path: Path to session_memory.json
        
    Returns:
        Dictionary containing session state
    """
    if json_path.exists():
        with open(json_path, 'r', encoding='utf-8') as f:
            memory = json.load(f)
    else:
        memory = SESSION_MEMORY_SCHEMA.copy()
    
    return memory

def save_session_memory(memory: Dict[str, Any], json_path: Path) -> None:
    """
    Persist session memory to JSON file.
    
    Args:
        memory: Session state dictionary
        json_path: Path to session_memory.json
    """
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(memory, f, indent=2)

# Initialize session memory
session_memory_path = exports_dir / 'session_memory.json'
session_memory = load_session_memory(session_memory_path)

print(f"\nSession memory initialized:")
print(f"  Profile: {session_memory.get('patient_profile', '(empty)')}")
print(f"  Turn count: {session_memory.get('turn_count', 0)}")
print(f"  Keywords: {session_memory.get('keywords', [])}")

## Section 2: Define PCDC Prompt Template with VIPS Framework

In [ ]:
# PCDC Prompt Template (VIPS Framework, ~180 tokens)
# This template integrates:
# - VALUE: Acknowledge patient's history and interests
# - INDIVIDUALIZE: Reference specific memories/keywords
# - PERSPECTIVE: Interpret behavior through person's lens
# - SOCIAL PSYCHOLOGY: Use engagement principles to suggest nudges

PCDC_PROMPT_TEMPLATE = """PCDC Memory Context
Patient: {patient_profile}
Session: {session_summary}
Keywords: {keywords_str}
Prior Nudge: {last_nudge}

Live Audio (15-20s)
Patient: "{patient_transcript}"
Carer: "{carer_transcript}"
Biomarkers: {biomarkers_str}

VIPS Task
Based on VALUE (listen), INDIVIDUALIZE (their story), PERSPECTIVE (their viewpoint), and SOCIAL psychology:
1. RATE agitation (0=calm to 10=severe)
2. TREND: one phrase describing change vs prior
3. KEYWORDS: max 3 key topics mentioned (update if new)
4. NUDGES: 1-2 PCDC responses citing their interests/memories

Output ONLY valid JSON:
{"agitation":_,"trend":"_","keywords":["",""],"nudges":["_","_"]}"""

def construct_pcdc_prompt(
    patient_profile: str,
    session_summary: str,
    keywords: List[str],
    last_nudge: str,
    patient_transcript: str,
    carer_transcript: str,
    biomarkers: Dict[str, float]
) -> str:
    """
    Construct the PCDC prompt with dynamic content injection.
    
    This implements the VIPS framework:
    - VALUE: Prompt refers to patient's profile and keywords (their values)
    - INDIVIDUALIZE: Session summary personalizes the context
    - PERSPECTIVE: Asks model to interpret from patient's viewpoint
    - SOCIAL PSYCHOLOGY: Requests PCDC-aligned nudges
    
    Args:
        patient_profile: Patient demographics and interests
        session_summary: Summary of session so far
        keywords: Current keywords of interest
        last_nudge: Previous nudge provided
        patient_transcript: Recent speech from patient
        carer_transcript: Recent speech from carer
        biomarkers: Dictionary of acoustic features from Phase 1.4
        
    Returns:
        Complete prompt string ready for model inference
    """
    # Format keywords as comma-separated string
    keywords_str = ", ".join(keywords) if keywords else "(none yet)"
    
    # Format biomarkers as feature:value pairs
    biomarkers_str = "; ".join(
        [f"{k}:{v:.2f}" for k, v in list(biomarkers.items())[:3]]  # Top 3 features
    )
    
    # Escape quotes in transcripts to prevent JSON injection
    patient_transcript = patient_transcript.replace('"', '\\"')
    carer_transcript = carer_transcript.replace('"', '\\"')
    
    prompt = PCDC_PROMPT_TEMPLATE.format(
        patient_profile=patient_profile,
        session_summary=session_summary,
        keywords_str=keywords_str,
        last_nudge=last_nudge,
        patient_transcript=patient_transcript,
        carer_transcript=carer_transcript,
        biomarkers_str=biomarkers_str
    )
    
    return prompt

# Example prompt construction
example_biomarkers = {
    "loudness_mean": 0.3,
    "intensity_score": 3,
    "voiced_segments_per_sec": 2.5,
    "articulation_variability": 0.54,
    "spectral_tilt": 0.01
}

example_prompt = construct_pcdc_prompt(
    patient_profile="Mary, 82, widow, loves gardening and bridge games",
    session_summary="15min into session: initially anxious about hospital visit, calmed when discussing rose garden",
    keywords=["roses", "garden", "hospital"],
    last_nudge="You've grown beautiful roses for 40 years - bridging to discuss how she managed gardens.",
    patient_transcript="I'm worried they'll take me to the hospital again.",
    carer_transcript="Mary, remember when we planted those roses? They're blooming beautifully now.",
    biomarkers=example_biomarkers
)

print("Example PCDC Prompt:")
print("=" * 70)
print(example_prompt)
print("=" * 70)
print(f"Prompt length: {len(example_prompt)} characters")

## Section 3: Load and Initialize MedGemma Model

In [ ]:
# Model Configuration for deterministic, reproducible inference
# MedGemma: Medical domain fine-tune of Gemma, specialized for clinical decision support

MODEL_CONFIG = {
    "model_name": "medgemma-1.5-4b",
    "model_path": str(assets_dir / "medgemma-1.5-4b"),
    "max_new_tokens": 120,
    "temperature": 0.0,  # Deterministic (no sampling)
    "do_sample": False,   # Required for deterministic output
    "top_p": 1.0,
    "top_k": None,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

print("Loading MedGemma model...")
print(f"Model path: {MODEL_CONFIG['model_path']}")

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_CONFIG['model_path'],
        trust_remote_code=True,
        local_files_only=True
    )
    
    # Load model with optimizations for local inference
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_CONFIG['model_path'],
        trust_remote_code=True,
        local_files_only=True,
        device_map=MODEL_CONFIG['device'],
        torch_dtype=torch.float32,  # Use float32 for accuracy (will quantize for mobile later)
        low_cpu_mem_usage=True
    )
    
    # Move to device if not already mapped
    if MODEL_CONFIG['device'] == 'cuda' and torch.cuda.is_available():
        model = model.to(MODEL_CONFIG['device'])
    
    model.eval()  # Set to evaluation mode
    
    print(f"✓ Model loaded successfully")
    print(f"  Device: {MODEL_CONFIG['device']}")
    print(f"  Model dtype: {model.dtype}")
    print(f"  Model parameters: {model.num_parameters() / 1e9:.2f}B")
    
except Exception as e:
    print(f"✗ Error loading model: {e}")
    print("Note: For Phase 2 development, model inference can be mocked.")

# Create inference pipeline
def generate_pcdc_response(prompt: str, model, tokenizer, config: Dict[str, Any]) -> str:
    """
    Generate PCDC response using MedGemma with deterministic inference.
    
    Key parameters:
    - do_sample=False: Ensures deterministic output (same input always produces same output)
    - max_new_tokens=120: Keeps responses concise for UI
    - temperature=0.0: Further enforces greedy decoding
    
    This determinism is critical for:
    1. Reproducibility: Same turn always gets same nudge
    2. Testing: Can validate VIPS adherence reliably
    3. Mobile: Consistent behavior across devices
    
    Args:
        prompt: PCDC prompt template with injected context
        model: MedGemma model instance
        tokenizer: Associated tokenizer
        config: Model configuration dictionary
        
    Returns:
        Generated response string (expected to be JSON)
    """
    inputs = tokenizer(prompt, return_tensors='pt')
    
    if config['device'] == 'cuda':
        inputs = {k: v.to(config['device']) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config['max_new_tokens'],
            temperature=config['temperature'],
            do_sample=config['do_sample'],
            top_p=config['top_p'],
            top_k=config['top_k']
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the generated part (after the input)
    response = response[len(prompt):].strip()
    
    return response

print("\n✓ Inference pipeline ready")

## Section 4: Implement Rolling Memory Management

In [ ]:
def extract_keywords(transcript: str, current_keywords: List[str], max_keywords: int = 3) -> List[str]:
    """
    Extract meaningful keywords from transcript (simplified heuristic).
    
    In production, this could use NER or semantic similarity, but for Phase 2
    we keep it minimal: capitalize/proper nouns likely topics.
    
    Args:
        transcript: Text from patient speech
        current_keywords: Existing keywords to preserve
        max_keywords: Maximum keywords to maintain
        
    Returns:
        Updated keyword list
    """
    # Preserve current keywords
    keywords = current_keywords.copy()
    
    # Simple heuristic: split by common delimiters, look for capitalized words
    words = transcript.split()
    for word in words:
        if len(word) > 4 and word[0].isupper() and word not in keywords:
            # Normalize (remove punctuation, convert to lowercase)
            normalized = word.rstrip(',.!?').lower()
            if normalized not in keywords and len(keywords) < max_keywords:
                keywords.append(normalized)
    
    return keywords[:max_keywords]

def estimate_agitation_trend(
    new_agitation: int,
    agitation_history: List[int],
    window_size: int = 3
) -> str:
    """
    Estimate trend in agitation based on recent history.
    
    Clinical interpretation:
    - Stable: agitation within 1 point of baseline
    - Calming: decreasing agitation trend
    - Escalating: increasing agitation trend
    
    Args:
        new_agitation: Latest agitation score (0-10)
        agitation_history: Previous agitation scores
        window_size: Number of recent scores to consider
        
    Returns:
        Trend phrase (e.g., "stabilizing_calm", "escalating_agitation")
    """
    if not agitation_history:
        return "baseline_established"
    
    recent = agitation_history[-window_size:]
    baseline = sum(recent) / len(recent)
    
    # Determine trend
    if new_agitation < baseline - 1:
        trend = f"calming_from_{int(baseline)}"
    elif new_agitation > baseline + 1:
        trend = f"escalating_to_{new_agitation}"
    else:
        trend = f"stable_{int(baseline)}"
    
    return trend

def summarize_session(
    keywords: List[str],
    agitation_history: List[int],
    prior_summary: str,
    turn_count: int
) -> str:
    """
    Auto-summarize session to prevent context dilution.
    
    Called every 5 turns to compress session info into ~30-50 words.
    Rationale: Prevents LLM from losing track over long sessions.
    
    Args:
        keywords: Current keywords
        agitation_history: All agitation scores so far
        prior_summary: Previous session summary
        turn_count: Total turns completed
        
    Returns:
        Concise session summary
    """
    if not agitation_history:
        return "Session started"
    
    avg_agitation = sum(agitation_history) / len(agitation_history)
    trend = "calming" if agitation_history[-1] < avg_agitation else "escalating"
    keywords_str = ", ".join(keywords)
    
    summary = f"{turn_count}min: {trend} (avg agitation {avg_agitation:.1f}); keywords: {keywords_str}"
    return summary

def update_memory(
    memory: Dict[str, Any],
    agitation_score: int,
    trend: str,
    keywords: List[str],
    nudge: str,
    turn_count: int
) -> Dict[str, Any]:
    """
    Update session memory with turn results.
    
    Triggers auto-summarization every 5 turns to maintain context freshness.
    
    Args:
        memory: Current session memory
        agitation_score: Latest agitation rating (0-10)
        trend: Calculated trend phrase
        keywords: Updated keyword list
        nudge: Generated PCDC nudge
        turn_count: Incremented turn counter
        
    Returns:
        Updated memory dictionary
    """
    # Maintain rolling agitation history (keep last 10 scores for trend)
    memory['agritation_history'].append(agitation_score)
    if len(memory['agritation_history']) > 10:
        memory['agritation_history'] = memory['agritation_history'][-10:]
    
    # Update mutable fields
    memory['current_trend'] = trend
    memory['keywords'] = keywords
    memory['last_nudge'] = nudge
    memory['turn_count'] = turn_count
    
    # Auto-summarize every 5 turns
    if turn_count % 5 == 0:
        memory['session_summary'] = summarize_session(
            keywords,
            memory['agritation_history'],
            memory['session_summary'],
            turn_count
        )
    
    return memory

# Test memory functions
test_memory = load_session_memory(session_memory_path)
test_memory['patient_profile'] = "Mary, 82, widow, loves gardening"
test_memory['agritation_history'] = [5, 4, 3, 4, 5]

keywords = extract_keywords("I planted roses in the garden yesterday", [], max_keywords=3)
print(f"Extracted keywords: {keywords}")

trend = estimate_agitation_trend(3, [5, 4, 5, 6, 5])
print(f"Agitation trend: {trend}")

summary = summarize_session(keywords, [5, 4, 3, 4, 5], "", 5)
print(f"Session summary: {summary}")

print("\n✓ Memory management functions loaded")

## Section 5: Create PCDC Inference Pipeline

In [ ]:
# Fallback PCDC nudges for when JSON parsing fails
# These are manually crafted to embody VIPS principles
PCDC_FALLBACK_NUDGES = [
    "Tell me more about what matters most to you.",
    "I notice you seem [emotional_state]. Let's take a moment.",
    "What was a happy time you remember?",
    "I'm here to listen to your perspective.",
    "Let's focus on what brings you comfort.",
    "Your feelings are valid and important.",
    "I understand this is difficult for you.",
    "What would help you feel safer right now?"
]

def run_pcdc_inference(
    patient_transcript: str,
    carer_transcript: str,
    biomarkers: Dict[str, float],
    memory: Dict[str, Any],
    model=None,
    tokenizer=None,
    use_mock: bool = False
) -> Tuple[Dict[str, Any], str]:
    """
    End-to-end PCDC inference pipeline.
    
    This orchestrates:
    1. Memory loading
    2. Prompt construction
    3. Model generation
    4. JSON parsing and validation
    5. Memory update and persistence
    
    Args:
        patient_transcript: Patient's recent speech
        carer_transcript: Carer's recent speech
        biomarkers: Acoustic features from Phase 1.4
        memory: Current session memory
        model: MedGemma model (can be None for mocking)
        tokenizer: Associated tokenizer
        use_mock: If True, return mock response (for testing without model)
        
    Returns:
        Tuple of (parsed_output, raw_response) where:
        - parsed_output: Dict with agitation, trend, keywords, nudges
        - raw_response: Raw model output (for debugging)
    """
    # Construct PCDC prompt
    prompt = construct_pcdc_prompt(
        patient_profile=memory.get('patient_profile', ''),
        session_summary=memory.get('session_summary', ''),
        keywords=memory.get('keywords', []),
        last_nudge=memory.get('last_nudge', ''),
        patient_transcript=patient_transcript,
        carer_transcript=carer_transcript,
        biomarkers=biomarkers
    )
    
    # Generate response
    if use_mock or model is None:
        # Mock response for development without model
        raw_response = '{"agitation":5,"trend":"stable","keywords":["memory"],"nudges":["Tell me about your day."]}'
    else:
        raw_response = generate_pcdc_response(prompt, model, tokenizer, MODEL_CONFIG)
    
    # Parse and validate JSON
    parsed_output, _ = parse_and_validate_json(raw_response, prompt)
    
    # Update memory with results
    memory = update_memory(
        memory=memory,
        agitation_score=parsed_output.get('agitation', 5),
        trend=parsed_output.get('trend', 'stable'),
        keywords=parsed_output.get('keywords', []),
        nudge=parsed_output.get('nudges', [''])[0],
        turn_count=memory.get('turn_count', 0) + 1
    )
    
    # Persist to disk
    save_session_memory(memory, session_memory_path)
    
    return parsed_output, raw_response

print("✓ PCDC inference pipeline ready")

## Section 6: Parse and Validate JSON Output

In [ ]:
def parse_and_validate_json(raw_response: str, prompt: str) -> Tuple[Dict[str, Any], bool]:
    """
    Parse JSON output from MedGemma and validate schema.
    
    Expected JSON structure:
    {
        "agitation": 0-10 (int),
        "trend": "phrase" (str),
        "keywords": ["kw1", "kw2", "kw3"] (list, max 3),
        "nudges": ["nudge1", "nudge2"] (list, 1-2 items)
    }
    
    Validation rules:
    - agitation: Must be 0-10 integer
    - trend: Must be non-empty string, max 50 chars
    - keywords: Max 3 keywords, each < 20 chars
    - nudges: 1-2 nudges, each < 100 chars, must embed PCDC principles
    
    On validation failure, returns fallback response with appropriate nudge.
    
    Args:
        raw_response: Raw model output
        prompt: Original prompt (for debugging)
        
    Returns:
        Tuple of (validated_dict, is_valid_flag)
    """
    
    # Default fallback response (PCDC-aligned)
    fallback = {
        "agitation": 5,
        "trend": "baseline",
        "keywords": [],
        "nudges": ["I'm here to listen. What's on your mind?"]
    }
    
    try:
        # Try to extract JSON from response
        # Handle case where model outputs extra text before/after JSON
        start_idx = raw_response.find('{')
        end_idx = raw_response.rfind('}') + 1
        
        if start_idx == -1 or end_idx <= start_idx:
            print("Warning: No JSON found in response")
            return fallback, False
        
        json_str = raw_response[start_idx:end_idx]
        output = json.loads(json_str)
        
    except json.JSONDecodeError as e:
        print(f"Warning: JSON parse error: {e}")
        return fallback, False
    
    # Validate and sanitize fields
    try:
        # Validate agitation
        agitation = int(output.get('agitation', 5))
        if not (0 <= agitation <= 10):
            agitation = max(0, min(10, agitation))  # Clamp to range
        
        # Validate trend
        trend = str(output.get('trend', 'stable'))[:50]  # Max 50 chars
        if not trend:
            trend = 'stable'
        
        # Validate keywords
        keywords_raw = output.get('keywords', [])
        if not isinstance(keywords_raw, list):
            keywords_raw = []
        keywords = [str(kw)[:20] for kw in keywords_raw[:3]]  # Max 3, each < 20 chars
        
        # Validate nudges
        nudges_raw = output.get('nudges', [])
        if not isinstance(nudges_raw, list):
            nudges_raw = []
        nudges = [str(n)[:100] for n in nudges_raw[:2]]  # Max 2, each < 100 chars
        if not nudges:
            nudges = ["I'm here to support you. Tell me how you're feeling."]
        
        # Construct validated output
        validated = {
            "agitation": agitation,
            "trend": trend,
            "keywords": keywords,
            "nudges": nudges
        }
        
        return validated, True
        
    except Exception as e:
        print(f"Warning: Validation error: {e}")
        return fallback, False

# Test JSON parsing
test_json_responses = [
    '{"agitation":7,"trend":"escalating anxiety","keywords":["hospital","worried"],"nudges":["You\'re safe here.","Let\'s focus on the present."]}',
    '{bad json}',
    '{"agitation":"high"}',  # Missing fields
]

print("Testing JSON Parsing:")
print("=" * 70)
for i, test_response in enumerate(test_json_responses):
    parsed, is_valid = parse_and_validate_json(test_response, "dummy_prompt")
    print(f"Test {i+1}: Valid={is_valid}")
    print(f"  Output: {parsed}")
    print()

print("✓ JSON validation functions loaded")

## Section 7: Test with Synthetic Turns and Score VIPS Adherence

In [ ]:
def score_vips_adherence(nudges: List[str]) -> float:
    """
    Score a list of nudges against VIPS framework criteria.
    
    VIPS Criteria (each 0-25 points):
    - VALUE: Does nudge acknowledge patient's values/interests? (look for personalization)
    - INDIVIDUALIZE: Is it tailored to this specific person? (keywords present)
    - PERSPECTIVE: Does it honor the patient's viewpoint? (validation language)
    - SOCIAL PSYCHOLOGY: Does it use engagement principles? (warm, non-confrontational)
    
    Total: 0-100 score (target >= 90%)
    
    Args:
        nudges: List of nudge strings from model output
        
    Returns:
        Score 0-100
    """
    score = 0
    
    for nudge in nudges:
        nudge_lower = nudge.lower()
        
        # VALUE (personalization, interest acknowledgement)
        value_keywords = ['garden', 'roses', 'loved', 'enjoyed', 'remember', 'special']
        if any(kw in nudge_lower for kw in value_keywords):
            score += 15
        else:
            score += 5  # Partial credit for trying
        
        # INDIVIDUALIZE (specific to person, not generic)
        individualize_keywords = ['your', 'you', 'name', 'daughter', 'son', 'family']
        if any(kw in nudge_lower for kw in individualize_keywords):
            score += 15
        else:
            score += 5
        
        # PERSPECTIVE (validation, non-confrontational)
        perspective_keywords = ['understand', 'feel', 'valid', 'sense', 'perspective', 'important']
        if any(kw in nudge_lower for kw in perspective_keywords):
            score += 15
        else:
            score += 5
        
        # SOCIAL PSYCHOLOGY (warm engagement, affiliation not authority)
        social_keywords = ['together', 'here', 'support', 'help', 'listen', 'moment', 'calm']
        if any(kw in nudge_lower for kw in social_keywords):
            score += 15
        else:
            score += 5
        
        # Penalty for confrontational language
        confrontation_keywords = ['no', 'dont', 'cannot', 'wrong', 'stop', 'must']
        if any(kw in nudge_lower for kw in confrontation_keywords):
            score -= 15  # Significant penalty
    
    # Average across nudges, cap at 100
    if nudges:
        final_score = score / len(nudges)
    else:
        final_score = 0
    
    return min(100, max(0, final_score))

# Synthetic test scenarios for VIPS validation
SYNTHETIC_TEST_SCENARIOS = [
    {
        "name": "Calm baseline",
        "patient_transcript": "Good morning. How are you today?",
        "carer_transcript": "I'm doing well. How did you sleep?",
        "biomarkers": {"loudness_mean": 0.4, "intensity_score": 2},
        "expected_agitation": 2
    },
    {
        "name": "Mild anxiety (hospital mention)",
        "patient_transcript": "I'm worried about the next doctor visit.",
        "carer_transcript": "Let's talk about that. What concerns you?",
        "biomarkers": {"loudness_mean": 0.25, "intensity_score": 4},
        "expected_agitation": 5
    },
    {
        "name": "Escalating agitation",
        "patient_transcript": "I can't find my keys! Why is everything always moving?!",
        "carer_transcript": "Let's look together. Your keys are here.",
        "biomarkers": {"loudness_mean": 0.15, "intensity_score": 7},
        "expected_agitation": 7
    },
    {
        "name": "Garden interest",
        "patient_transcript": "I remember planting roses in the spring.",
        "carer_transcript": "That's right! The roses are beautiful this year.",
        "biomarkers": {"loudness_mean": 0.5, "intensity_score": 2},
        "expected_agitation": 2
    }
]

# Run tests
print("\nSynthetic Turn Testing (VIPS Adherence Scoring)")
print("=" * 70)

vips_scores = []

for scenario in SYNTHETIC_TEST_SCENARIOS:
    print(f"\nScenario: {scenario['name']}")
    print("-" * 70)
    
    # Initialize memory for this test
    test_memory = {
        "patient_profile": "Test patient, interested in gardening",
        "session_summary": "Initial session",
        "current_trend": "baseline",
        "keywords": ["garden", "roses"],
        "last_nudge": "Welcome to our session.",
        "agritation_history": [],
        "turn_count": 0
    }
    
    # Run inference (mock mode)
    output, raw_response = run_pcdc_inference(
        patient_transcript=scenario["patient_transcript"],
        carer_transcript=scenario["carer_transcript"],
        biomarkers=scenario["biomarkers"],
        memory=test_memory,
        model=None,
        tokenizer=None,
        use_mock=True  # Use mock for Phase 2 dev
    )
    
    # Score VIPS adherence
    vips_score = score_vips_adherence(output.get("nudges", []))
    vips_scores.append(vips_score)
    
    print(f"Agitation: {output['agitation']}/10 (expected ~{scenario['expected_agitation']})")
    print(f"Trend: {output['trend']}")
    print(f"Keywords: {output['keywords']}")
    print(f"Nudge: {output['nudges'][0]}")
    print(f"VIPS Score: {vips_score:.1f}/100")

# Summary statistics
avg_vips = sum(vips_scores) / len(vips_scores) if vips_scores else 0
print(f"\n{'=' * 70}")
print(f"VIPS Adherence Validation Summary")
print(f"{'=' * 70}")
print(f"Average VIPS Score: {avg_vips:.1f}/100")
print(f"Target: >= 90%")
print(f"Status: {'✓ PASS' if avg_vips >= 90 else '✗ NEEDS TUNING'}")

print("\n✓ Phase 2 Development Complete")